# NER and Entity Resolution - spaCy + EntityRuler
This work is a part of the knowledge graph pipeline built for 26 Spring BigCo Studio at Cornell Tech, and belongs to Team 10.

The whole process of this part is run on Google Colab.

# 1. Set up GDrive

In [30]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [31]:
%cd /content/gdrive/MyDrive/NER_EntityResolution

/content/gdrive/MyDrive/NER_EntityResolution


# 2. Install required packages

In [32]:
!pip install -q pandas spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 66.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# 3. Preprocess data

In [33]:
!python src/preprocess.py \
  --input data/raw/stage1_records_merged.csv \
  --cleaned-output data/processed/cleaned_input_stage1_merged.csv \
  --invalid-output data/processed/invalid_rows_stage1_merged.csv

[preprocess] Wrote 2782 valid row(s) to: data/processed/cleaned_input_stage1_merged.csv
[preprocess] Wrote 0 invalid row(s) to: data/processed/invalid_rows_stage1_merged.csv


In [34]:
!ls data/processed/

cleaned_input_mock1.csv		 invalid_rows_mock1.csv
cleaned_input_mock2.csv		 invalid_rows_mock2.csv
cleaned_input_stage1.csv	 invalid_rows_stage1.csv
cleaned_input_stage1_merged.csv  invalid_rows_stage1_merged.csv


In [35]:
import pandas as pd

cleaned_df = pd.read_csv("data/processed/cleaned_input_stage1_merged.csv")
invalid_df = pd.read_csv("data/processed/invalid_rows_stage1_merged.csv")

print("Cleaned input:")
display(cleaned_df)

print("Invalid rows:")
display(invalid_df)

Cleaned input:


,source_id,source_type,raw_text,source_url,date,company_seed,source_type_cleaned,raw_text_cleaned,source_url_cleaned,date_cleaned,company_seed_cleaned,company_seed_invalid,company_seed_invalid_reason,row_invalid_reasons,row_valid_for_ner
0,1,USPTO,Patent 10538523. Assignee: PFIZER INC.. Grant ...,https://patents.google.com/patent/US10538523,2020,Pfizer,USPTO,Patent 10538523. Assignee: PFIZER INC.. Grant ...,https://patents.google.com/patent/US10538523,2020,Pfizer,False,NaN,NaN,True
1,2,USPTO,Patent 10538542. Assignee: PFIZER INC.. Grant ...,https://patents.google.com/patent/US10538542,2020,Pfizer,USPTO,Patent 10538542. Assignee: PFIZER INC.. Grant ...,https://patents.google.com/patent/US10538542,2020,Pfizer,False,NaN,NaN,True
2,3,USPTO,Patent 10543267. Assignee: PFIZER INC.. Grant ...,https://patents.google.com/patent/US10543267,2020,Pfizer,USPTO,Patent 10543267. Assignee: PFIZER INC.. Grant ...,https://patents.google.com/patent/US10543267,2020,Pfizer,False,NaN,NaN,True
3,4,USPTO,Patent 10544095. Assignee: PFIZER INC.. Grant ...,https://patents.google.com/patent/US10544095,2020,Pfizer,USPTO,Patent 10544095. Assignee: PFIZER INC.. Grant ...,https://patents.google.com/patent/US10544095,2020,Pfizer,False,NaN,NaN,True
4,5,USPTO,Patent 10544212. Assignee: PFIZER INC.. Grant ...,https://patents.google.com/patent/US10544212,2020,Pfizer,USPTO,Patent 10544212. Assignee: PFIZER INC.. Grant ...,https://patents.google.com/patent/US10544212,2020,Pfizer,False,NaN,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2777,2778,PUBMED,Antihypertensive Deprescribing and Cardiovascu...,https://pubmed.ncbi.nlm.nih.gov/39585693/,Nov-24,VA PALO ALTO HEALTH CARE SYSTEM,PUBMED,Antihypertensive Deprescribing and Cardiovascu...,https://pubmed.ncbi.nlm.nih.gov/39585693/,Nov-24,VA PALO ALTO HEALTH CARE SYSTEM,False,NaN,NaN,True
2778,2779,PUBMED,Epstein-Barr virus reprograms autoreactive B c...,https://pubmed.ncbi.nlm.nih.gov/41223250/,Nov-25,VA PALO ALTO HEALTH CARE SYSTEM,PUBMED,Epstein-Barr virus reprograms autoreactive B c...,https://pubmed.ncbi.nlm.nih.gov/41223250/,Nov-25,VA PALO ALTO HEALTH CARE SYSTEM,False,NaN,NaN,True
2779,2780,PUBMED,Information and resources VA health system lea...,https://pubmed.ncbi.nlm.nih.gov/39073213/,Oct-24,VA PALO ALTO HEALTH CARE SYSTEM,PUBMED,Information and resources VA health system lea...,https://pubmed.ncbi.nlm.nih.gov/39073213/,Oct-24,VA PALO ALTO HEALTH CARE SYSTEM,False,NaN,NaN,True
2780,2781,PUBMED,Large Language Model Influence on Diagnostic R...,https://pubmed.ncbi.nlm.nih.gov/39466245/,Oct-24,VA PALO ALTO HEALTH CARE SYSTEM,PUBMED,Large Language Model Influence on Diagnostic R...,https://pubmed.ncbi.nlm.nih.gov/39466245/,Oct-24,VA PALO ALTO HEALTH CARE SYSTEM,False,NaN,NaN,True


Invalid rows:


,source_id,source_type,raw_text,source_url,date,company_seed,source_type_cleaned,raw_text_cleaned,source_url_cleaned,date_cleaned,company_seed_cleaned,company_seed_invalid,company_seed_invalid_reason,row_invalid_reasons,row_valid_for_ner


# 4. Run spaCy + EntityRuler for NER

In [36]:
!python src/ner_spacy_entityruler.py \
  --input data/processed/cleaned_input_stage1_merged.csv \
  --output outputs/ner/ner_results_entityruler_stage1_merged.csv \
  --spacy-model en_core_web_sm

[ner_spacy_entityruler] Wrote 11251 mention(s) to: outputs/ner/ner_results_entityruler_stage1_merged.csv


In [37]:
ner_df = pd.read_csv("outputs/ner/ner_results_entityruler_stage1_merged.csv")
display(ner_df)

,source_id,source_type,source_type_cleaned,source_url,source_url_cleaned,date,date_cleaned,company_seed,company_seed_cleaned,raw_text,raw_text_cleaned,raw_mention,entity_label,start_char,end_char,mention_source,mention_confidence
0,1,USPTO,USPTO,https://patents.google.com/patent/US10538523,https://patents.google.com/patent/US10538523,2020,2020,Pfizer,Pfizer,Patent 10538523. Assignee: PFIZER INC.. Grant ...,Patent 10538523. Assignee: PFIZER INC.. Grant ...,PFIZER,ORG,27,33,spacy_or_entityruler,NaN
1,1,USPTO,USPTO,https://patents.google.com/patent/US10538523,https://patents.google.com/patent/US10538523,2020,2020,Pfizer,Pfizer,Patent 10538523. Assignee: PFIZER INC.. Grant ...,Patent 10538523. Assignee: PFIZER INC.. Grant ...,CPC,ORG,141,144,spacy_or_entityruler,NaN
2,2,USPTO,USPTO,https://patents.google.com/patent/US10538542,https://patents.google.com/patent/US10538542,2020,2020,Pfizer,Pfizer,Patent 10538542. Assignee: PFIZER INC.. Grant ...,Patent 10538542. Assignee: PFIZER INC.. Grant ...,PFIZER,ORG,27,33,spacy_or_entityruler,NaN
3,2,USPTO,USPTO,https://patents.google.com/patent/US10538542,https://patents.google.com/patent/US10538542,2020,2020,Pfizer,Pfizer,Patent 10538542. Assignee: PFIZER INC.. Grant ...,Patent 10538542. Assignee: PFIZER INC.. Grant ...,CPC,ORG,140,143,spacy_or_entityruler,NaN
4,3,USPTO,USPTO,https://patents.google.com/patent/US10543267,https://patents.google.com/patent/US10543267,2020,2020,Pfizer,Pfizer,Patent 10543267. Assignee: PFIZER INC.. Grant ...,Patent 10543267. Assignee: PFIZER INC.. Grant ...,PFIZER,ORG,27,33,spacy_or_entityruler,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11246,2781,PUBMED,PUBMED,https://pubmed.ncbi.nlm.nih.gov/39466245/,https://pubmed.ncbi.nlm.nih.gov/39466245/,Oct-24,Oct-24,VA PALO ALTO HEALTH CARE SYSTEM,VA PALO ALTO HEALTH CARE SYSTEM,Large Language Model Influence on Diagnostic R...,Large Language Model Influence on Diagnostic R...,CI,ORG,2233,2235,spacy_or_entityruler,NaN
11247,2781,PUBMED,PUBMED,https://pubmed.ncbi.nlm.nih.gov/39466245/,https://pubmed.ncbi.nlm.nih.gov/39466245/,Oct-24,Oct-24,VA PALO ALTO HEALTH CARE SYSTEM,VA PALO ALTO HEALTH CARE SYSTEM,Large Language Model Influence on Diagnostic R...,Large Language Model Influence on Diagnostic R...,LLM,ORG,2354,2357,spacy_or_entityruler,NaN
11248,2781,PUBMED,PUBMED,https://pubmed.ncbi.nlm.nih.gov/39466245/,https://pubmed.ncbi.nlm.nih.gov/39466245/,Oct-24,Oct-24,VA PALO ALTO HEALTH CARE SYSTEM,VA PALO ALTO HEALTH CARE SYSTEM,Large Language Model Influence on Diagnostic R...,Large Language Model Influence on Diagnostic R...,LLM,ORG,2483,2486,spacy_or_entityruler,NaN
11249,2782,PUBMED,PUBMED,https://pubmed.ncbi.nlm.nih.gov/40899690/,https://pubmed.ncbi.nlm.nih.gov/40899690/,Sep-25,Sep-25,VA PALO ALTO HEALTH CARE SYSTEM,VA PALO ALTO HEALTH CARE SYSTEM,ACG Clinical Guideline: Perioperative Risk Ass...,ACG Clinical Guideline: Perioperative Risk Ass...,ACG,ORG,0,3,spacy_or_entityruler,NaN


# 5. Run alias resolution

In [38]:
!python src/resolve_alias.py \
  --input outputs/ner/ner_results_entityruler_stage1_merged.csv \
  --resolved-output outputs/entity_resolution/resolved_entities_entityruler_stage1_merged.csv \
  --review-output outputs/entity_resolution/review_queue_entityruler_stage1_merged.csv

[resolve_alias] Wrote 11251 resolved row(s) to: outputs/entity_resolution/resolved_entities_entityruler_stage1_merged.csv
[resolve_alias] Wrote 196 review pair(s) to: outputs/entity_resolution/review_queue_entityruler_stage1_merged.csv


In [39]:
import os

for path in [
    "outputs/entity_resolution/resolved_entities_entityruler_stage1_merged.csv",
    "outputs/entity_resolution/review_queue_entityruler_stage1_merged.csv",
]:
    print(path)
    print("exists:", os.path.exists(path))
    if os.path.exists(path):
        print("size:", os.path.getsize(path), "bytes")
    print("-" * 40)

outputs/entity_resolution/resolved_entities_entityruler_stage1_merged.csv
exists: True
size: 30060908 bytes
----------------------------------------
outputs/entity_resolution/review_queue_entityruler_stage1_merged.csv
exists: True
size: 29521 bytes
----------------------------------------


# 6. Candidate relations

In [40]:
import pandas as pd

resolved_df = pd.read_csv("outputs/entity_resolution/resolved_entities_entityruler_stage1_merged.csv")
display(resolved_df)

,source_id,source_type,source_url,date,company_seed,raw_text,raw_mention,entity_label,start_char,end_char,normalized_name,base_name,legal_suffixes,canonical_group_id,canonical_name,merge_confidence,merge_decision,evidence_note
0,1,USPTO,https://patents.google.com/patent/US10538523,2020,Pfizer,Patent 10538523. Assignee: PFIZER INC.. Grant ...,PFIZER,ORG,27,33,PFIZER,PFIZER,NaN,950,PFIZER,0.88,auto_merge,auto_merge_component
1,1,USPTO,https://patents.google.com/patent/US10538523,2020,Pfizer,Patent 10538523. Assignee: PFIZER INC.. Grant ...,CPC,ORG,141,144,CPC,CPC,NaN,698,CPC,1.00,singleton,singleton_no_merge_needed
2,2,USPTO,https://patents.google.com/patent/US10538542,2020,Pfizer,Patent 10538542. Assignee: PFIZER INC.. Grant ...,PFIZER,ORG,27,33,PFIZER,PFIZER,NaN,950,PFIZER,0.88,auto_merge,auto_merge_component
3,2,USPTO,https://patents.google.com/patent/US10538542,2020,Pfizer,Patent 10538542. Assignee: PFIZER INC.. Grant ...,CPC,ORG,140,143,CPC,CPC,NaN,698,CPC,1.00,singleton,singleton_no_merge_needed
4,3,USPTO,https://patents.google.com/patent/US10543267,2020,Pfizer,Patent 10543267. Assignee: PFIZER INC.. Grant ...,PFIZER,ORG,27,33,PFIZER,PFIZER,NaN,950,PFIZER,0.88,auto_merge,auto_merge_component
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11246,2781,PUBMED,https://pubmed.ncbi.nlm.nih.gov/39466245/,Oct-24,VA PALO ALTO HEALTH CARE SYSTEM,Large Language Model Influence on Diagnostic R...,CI,ORG,2233,2235,CI,CI,NaN,70,CI,1.00,singleton,singleton_no_merge_needed
11247,2781,PUBMED,https://pubmed.ncbi.nlm.nih.gov/39466245/,Oct-24,VA PALO ALTO HEALTH CARE SYSTEM,Large Language Model Influence on Diagnostic R...,LLM,ORG,2354,2357,LLM,LLM,NaN,908,LLM,1.00,singleton,singleton_no_merge_needed
11248,2781,PUBMED,https://pubmed.ncbi.nlm.nih.gov/39466245/,Oct-24,VA PALO ALTO HEALTH CARE SYSTEM,Large Language Model Influence on Diagnostic R...,LLM,ORG,2483,2486,LLM,LLM,NaN,908,LLM,1.00,singleton,singleton_no_merge_needed
11249,2782,PUBMED,https://pubmed.ncbi.nlm.nih.gov/40899690/,Sep-25,VA PALO ALTO HEALTH CARE SYSTEM,ACG Clinical Guideline: Perioperative Risk Ass...,ACG,ORG,0,3,ACG,ACG,NaN,1428,ACG,0.50,review_needed,high_string_similarity


In [41]:
!mkdir -p outputs/relations

!python src/extract_candidate_relations.py \
  --input outputs/entity_resolution/resolved_entities_entityruler_stage1_merged.csv \
  --output outputs/relations/candidate_relations_entityruler_stage1_merged.csv

[extract_candidate_relations] Wrote 52 candidate row(s) to: outputs/relations/candidate_relations_entityruler_stage1_merged.csv
